# BUS 32120, Week 3
# Adding to `groupby` and `transform`

## 1. Recall what `groupby` does: make groups!

`groupby` alone doesn't do much, it simply remembers which group an observation belongs to. Recall that to get anything useful out of groupby, you need to apply some kind of aggregation.

In [1]:
import pandas as pd

In [2]:
department_df = pd.DataFrame({'Department' : ['Marketing', 'HR', 'Marketing', 'HR',
                           'Marketing', 'HR'],
                   'City' : ['Chicago', 'Chicago', 'Seattle', 'Seattle',
                           'New York', 'New York'],
                   'People' : [12, 20, 31, 90, 120, 150],
                   'Revenue' :[200, 500, 300, 350, 1200, 3000]}, 
                             columns = ["Department", "City", "People", "Revenue"])

In [3]:
department_df

,Department,City,People,Revenue
0,Marketing,Chicago,12,200
1,HR,Chicago,20,500
2,Marketing,Seattle,31,300
3,HR,Seattle,90,350
4,Marketing,New York,120,1200
5,HR,New York,150,3000


### a) Create a `groupby` object

In [4]:
# create a groupby object 
grouped_dpt = department_df.groupby('Department')
grouped_dpt # groupby object 

**What exactly does a `groupby` object hold?**

In [5]:
grouped_dpt.groups # group names and their members

{'HR': [1, 3, 5], 'Marketing': [0, 2, 4]}

In [6]:
grouped_dpt.ngroups # number of groups

2

### b) Adding an aggregation to the `groupby` object

In [7]:
grouped_dpt.sum() # how we did this in the HW

,City,People,Revenue
Department,,,
HR,ChicagoSeattleNew York,260,3850
Marketing,ChicagoSeattleNew York,163,1700


### b.1) Using named aggregation with `pd.NamedAgg`

In [8]:
# aggregation: sum
grouped_dpt.aggregate(
    people_sum=pd.NamedAgg(column="People", aggfunc="sum"),
    revenue_sum=pd.NamedAgg(column="Revenue", aggfunc="sum"))

,people_sum,revenue_sum
Department,,
HR,260,3850
Marketing,163,1700


In [9]:
# aggregation: mean
grouped_dpt[['Revenue','People']].aggregate("mean")

,Revenue,People
Department,,
HR,1283.333333,86.666667
Marketing,566.666667,54.333333


In [10]:
# keep group key as a column with as_index = False
department_df.groupby("Department", as_index=False)[['Revenue','People']].mean()

,Department,Revenue,People
0,HR,1283.333333,86.666667
1,Marketing,566.666667,54.333333


In [11]:
# multiple groups
grouped_dpt_city = department_df.groupby(["Department", "City"])

In [12]:
grouped_dpt_city.aggregate("sum")

People  Revenue
Department City                     
HR         Chicago       20      500
           New York     150     3000
           Seattle       90      350
Marketing  Chicago       12      200
           New York     120     1200
           Seattle       31      300

### Re-do the groupby, this time adding `as_index = False`

Now department and city are columns in the output, not a multi-level index. This is often easier to work with.

In [13]:
department_df.groupby(["Department", "City"], as_index = False).sum()

,Department,City,People,Revenue
0,HR,Chicago,20,500
1,HR,New York,150,3000
2,HR,Seattle,90,350
3,Marketing,Chicago,12,200
4,Marketing,New York,120,1200
5,Marketing,Seattle,31,300


### c) Using `transform`

Unlike `groupby`, here `transform` returns a DF that's the **same shape as the original**, meaning the aggregated values (sum, median, etc) are repeated for several rows.

[Documentation here.](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.transform.html)

Create a new column, revenue_total, that is the revenue summed by department (notice using `grouped_dpt`)

In [15]:
department_df["Revenue_Total"] = grouped_dpt["Revenue"].transform("sum") # total revenue per department; same value is repeated for every city
department_df

,Department,City,People,Revenue,Revenue_Total
0,Marketing,Chicago,12,200,1700
1,HR,Chicago,20,500,3850
2,Marketing,Seattle,31,300,1700
3,HR,Seattle,90,350,3850
4,Marketing,New York,120,1200,1700
5,HR,New York,150,3000,3850


### d) Use case for `transform`: percentage of total 

In [16]:
department_df['pct_of_total_rev'] = department_df['Revenue'] / department_df['Revenue_Total']
department_df

,Department,City,People,Revenue,Revenue_Total,pct_of_total_rev
0,Marketing,Chicago,12,200,1700,0.117647
1,HR,Chicago,20,500,3850,0.129870
2,Marketing,Seattle,31,300,1700,0.176471
3,HR,Seattle,90,350,3850,0.090909
4,Marketing,New York,120,1200,1700,0.705882
5,HR,New York,150,3000,3850,0.779221
